# Run the whole pipeline in one go

This notebook executes the 21 analysis notebooks of stages 1, 2 and 3 in numeric order, each in its
own folder and its own kernel, and prints a status table at the end (notebook, seconds, passed or
failed, first error). It is the one file to run to regenerate every table in `output/` and every
figure in `figures/` from the raw files in `data/`. Expect about 4 minutes; Ripley's K (2_03) is the
slowest step at under a minute.

Nothing here needs the network:

- `1_05` serves every geocode from the committed cache `output/onemap_geocode_cache.csv`.
- `1_06` keeps `REBUILD_MATRIX = False`, so it summarises the saved walk matrices in `large_data/`
  instead of re-querying OSRM (about 1.7 hours). To rebuild, start `osrm-routed` and flip that flag
  inside `1_06` (see the stage 1 README).
- `2_04` draws a sample of 100 walking routes if an OSRM server answers on `localhost:5001`; if not,
  it says so and skips that one section. Everything else in it reads the saved matrices.

`0_Data_Exploration/` is not run: those notebooks are superseded experiments kept for the record.

Settings: `SAVE_NOTEBOOKS` writes each executed notebook back in place so its outputs are refreshed
(set it to `False` to only check that everything runs); `STOP_ON_ERROR` stops at the first failure;
`ONLY` runs a subset by prefix, e.g. `["3_06", "3_08"]`, for a quick re-run after a change.

In [1]:
from pathlib import Path
import time

import nbformat
import pandas as pd
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

ROOT = Path.cwd()
assert (ROOT / "src" / "bt4015").is_dir(), "run this notebook from the repository root"

STAGES = ["1_Data_Preparation", "2_ESDA", "3_Hypothesis_Testing"]
SAVE_NOTEBOOKS = True   # write each executed notebook back in place (outputs refreshed)
STOP_ON_ERROR = False   # True: stop at the first failing notebook; False: record it and carry on
TIMEOUT_S = 1800        # per cell; the slowest cell (Ripley's K simulations) takes about 40 s
ONLY = None             # e.g. ["3_06", "3_08"] to run a subset by filename prefix

# 1. Discover the pipeline notebooks: the numbered files of each stage, in numeric order.
notebooks = [path for stage in STAGES
             for path in sorted((ROOT / stage).glob("[0-9]_[0-9][0-9]_*.ipynb"))]
if ONLY:
    notebooks = [path for path in notebooks if any(path.name.startswith(prefix) for prefix in ONLY)]
print(f"{len(notebooks)} notebooks to run:")
for path in notebooks:
    print(f"  {path.relative_to(ROOT)}")

21 notebooks to run:
  1_Data_Preparation/1_01_load_subzones.ipynb
  1_Data_Preparation/1_02_join_hdb_blocks.ipynb
  1_Data_Preparation/1_03_cluster_hdb_blocks.ipynb
  1_Data_Preparation/1_04_assign_child_population.ipynb
  1_Data_Preparation/1_05_build_childcare_centres.ipynb
  1_Data_Preparation/1_06_build_walk_matrix.ipynb
  1_Data_Preparation/1_07_build_subzone_stats.ipynb
  1_Data_Preparation/1_08_excluded_centres_qa.ipynb
  2_ESDA/2_01_income_distribution.ipynb
  2_ESDA/2_02_child_density_kde.ipynb
  2_ESDA/2_03_ripleys_k.ipynb
  2_ESDA/2_04_walking_access.ipynb
  2_ESDA/2_05_centres_vs_children.ipynb
  3_Hypothesis_Testing/3_01_fee_spatial_autocorrelation.ipynb
  3_Hypothesis_Testing/3_02_income_fee_regression.ipynb
  3_Hypothesis_Testing/3_03_income_fee_gwr.ipynb
  3_Hypothesis_Testing/3_04_child_density_moran.ipynb
  3_Hypothesis_Testing/3_05_hc_bivariate_lisa.ipynb
  3_Hypothesis_Testing/3_06_pressure_index_lisa.ipynb
  3_Hypothesis_Testing/3_07_walking_access_by_ses.ipynb
  

## Run

Each notebook is executed with `nbclient` from inside its own stage folder (so its `ROOT` finder and
relative paths behave exactly as when it is opened by hand) in a fresh kernel, so no state leaks from
one notebook to the next. A failed notebook is not written back; its saved outputs stay as they were.

In [2]:
def run_notebook(path):
    """Execute one notebook in a fresh kernel, working directory = its own folder."""
    notebook = nbformat.read(path, as_version=4)
    client = NotebookClient(notebook, timeout=TIMEOUT_S, kernel_name="python3",
                            resources={"metadata": {"path": str(path.parent)}})
    client.execute()
    return notebook


rows = []
for path in notebooks:
    started = time.perf_counter()
    status, error = "ok", ""
    try:
        executed = run_notebook(path)
        if SAVE_NOTEBOOKS:
            nbformat.write(executed, path)
    except CellExecutionError as exc:            # a cell raised: keep the exception type and message
        status, error = "FAILED", f"{exc.ename}: {exc.evalue}"[:160]
    except Exception as exc:                      # kernel died, timeout, unreadable file
        status, error = "FAILED", f"{type(exc).__name__}: {exc}"[:160]
    seconds = time.perf_counter() - started
    rows.append((str(path.relative_to(ROOT)), path.parent.name, status, round(seconds, 1), error))
    print(f"{status:<6} {seconds:6.1f} s  {path.relative_to(ROOT)}  {error}")
    if status == "FAILED" and STOP_ON_ERROR:
        print("stopping at the first failure (STOP_ON_ERROR = True)")
        break

ok        1.6 s  1_Data_Preparation/1_01_load_subzones.ipynb  


ok        4.2 s  1_Data_Preparation/1_02_join_hdb_blocks.ipynb  


ok        3.6 s  1_Data_Preparation/1_03_cluster_hdb_blocks.ipynb  


ok        3.5 s  1_Data_Preparation/1_04_assign_child_population.ipynb  


ok        2.1 s  1_Data_Preparation/1_05_build_childcare_centres.ipynb  


ok        4.2 s  1_Data_Preparation/1_06_build_walk_matrix.ipynb  


... (11 outputs left out)

ok       14.1 s  3_Hypothesis_Testing/3_05_hc_bivariate_lisa.ipynb  


ok       15.1 s  3_Hypothesis_Testing/3_06_pressure_index_lisa.ipynb  


ok        2.7 s  3_Hypothesis_Testing/3_07_walking_access_by_ses.ipynb  


ok        3.4 s  3_Hypothesis_Testing/3_08_planned_centre_impact.ipynb  


## Status table

In [3]:
summary = pd.DataFrame(rows, columns=["notebook", "stage", "status", "seconds", "error"])
print(summary.to_string(index=False))
passed = int((summary["status"] == "ok").sum())
print(f"\n{passed} of {len(summary)} notebooks passed in {summary['seconds'].sum() / 60:.1f} min")
print("per stage:", summary.groupby("stage")["seconds"].sum().round(0).to_dict())
assert passed == len(summary), "some notebooks failed; see the error column"

                                                   notebook                stage status  seconds error
                1_Data_Preparation/1_01_load_subzones.ipynb   1_Data_Preparation     ok      1.6      
              1_Data_Preparation/1_02_join_hdb_blocks.ipynb   1_Data_Preparation     ok      4.2      
           1_Data_Preparation/1_03_cluster_hdb_blocks.ipynb   1_Data_Preparation     ok      3.6      
      1_Data_Preparation/1_04_assign_child_population.ipynb   1_Data_Preparation     ok      3.5      
      1_Data_Preparation/1_05_build_childcare_centres.ipynb   1_Data_Preparation     ok      2.1      
            1_Data_Preparation/1_06_build_walk_matrix.ipynb   1_Data_Preparation     ok      4.2      
          1_Data_Preparation/1_07_build_subzone_stats.ipynb   1_Data_Preparation     ok      1.9      
          1_Data_Preparation/1_08_excluded_centres_qa.ipynb   1_Data_Preparation     ok      7.0      
                      2_ESDA/2_01_income_distribution.ipynb              

## After a run

- Every file in `output/` and `figures/` has been regenerated; the stage READMEs list which notebook
  wrote which file.
- The headline numbers are printed at the end of each notebook (open it and scroll to the last cell),
  and summarised in the root README under "Hypotheses and headline results".
- Re-running is idempotent: the statistics are seeded (`SEED = 4015` throughout), so a second run
  reproduces the same tables and the same cluster labels.